# Name That LLM — DeBERTa + curated response corpus

This Kaggle notebook fine-tunes a public DeBERTa-v3-base checkpoint on the official examples plus the route-filtered public NTL corpus. It runs with internet disabled and writes `submission.parquet` for the code competition.

The external pool is attached as a Kaggle dataset; the pretrained weights are attached as a Kaggle model source. The official response rows receive extra loss weight to match the competition data.

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Read all competition data from Kaggle's mounted competition resource.
input_root = Path('/kaggle/input')
competition_dir = input_root / 'competitions/name-that-llm'
if not (competition_dir / 'train.csv').exists():
    competition_dir = next(p.parent for p in input_root.rglob('train.csv') if (p.parent / 'test.csv').exists())
train_df = pd.read_csv(competition_dir / 'train.csv')
test_df = pd.read_csv(competition_dir / 'test.csv')
external_path = next(input_root.rglob('curated_pool.parquet'))
external_df = pd.read_parquet(external_path)
print('official train/test:', train_df.shape, test_df.shape)
print('public corpus:', external_path, external_df.shape)

In [ ]:
# Confirm the corpus uses the same canonical model labels as the competition.
valid_models = sorted(train_df['model'].unique())
lookup = {m.casefold(): m for m in valid_models}
if not {'text', 'model'}.issubset(external_df.columns):
    raise ValueError(f'Unexpected external corpus columns: {external_df.columns.tolist()}')
ext = external_df[['text', 'model']].dropna().copy()
ext['model'] = ext['model'].astype(str).str.casefold().map(lookup)
ext['text'] = ext['text'].astype(str)
ext = ext.dropna(subset=['model'])
ext = ext[ext['text'].str.len().between(100, 3000)].reset_index(drop=True)
if set(ext['model'].unique()) != set(valid_models):
    raise ValueError(f'Missing external labels: {sorted(set(valid_models)-set(ext.model.unique()))}')
le = LabelEncoder().fit(valid_models)
y_external = le.transform(ext['model'])
y_train = le.transform(train_df['model'])
external_texts = ext['text'].tolist()
official_texts = train_df['text'].fillna('').astype(str).tolist()
test_texts = test_df['text'].fillna('').astype(str).tolist()
print('usable external rows:', len(ext), 'classes:', len(le.classes_))

In [ ]:
# Resolve the attached public Kaggle DeBERTa model locally (internet is disabled).
config_paths = [p for p in input_root.rglob('config.json') if 'deberta' in str(p.parent).casefold()]
if not config_paths:
    raise FileNotFoundError('The attached DeBERTa model source has no config.json')
model_dir = config_paths[0].parent
print('local model:', model_dir)
tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True, use_fast=True)


In [ ]:
# Train on the external corpus and official data. Give official examples extra
# weight because their response distribution matches the hidden test set.
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, source_weights=None):
        self.texts = texts
        self.labels = labels
        self.source_weights = source_weights
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        item = tokenizer(self.texts[i], truncation=True, max_length=384)
        if self.labels is not None:
            item['labels'] = int(self.labels[i])
            item['sample_weight'] = float(self.source_weights[i])
        return item

texts = external_texts + official_texts
labels = np.concatenate([y_external, y_train])
sample_weights = np.concatenate([np.ones(len(ext), dtype=np.float32),
                                 np.full(len(train_df), 5.0, dtype=np.float32)])
train_ds = TextDataset(texts, labels, sample_weights)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2,
                          pin_memory=torch.cuda.is_available(), collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(
    str(model_dir), num_labels=len(le.classes_), ignore_mismatched_sizes=True,
    local_files_only=True)
model.config.id2label = {i: name for i, name in enumerate(le.classes_)}
model.config.label2id = {name: i for i, name in enumerate(le.classes_)}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.float().to(device)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

counts = np.bincount(labels, minlength=len(le.classes_)).astype(np.float32)
class_weights = len(labels) / (len(le.classes_) * counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS = 3
ACCUM_STEPS = 1
steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.05 * steps_per_epoch * EPOCHS),
    num_training_steps=steps_per_epoch * EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader, 1):
        labels_batch = batch.pop('labels').to(device, non_blocking=True)
        row_weights = batch.pop('sample_weight').to(device, non_blocking=True)
        inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = model(**inputs, return_dict=False)[0]
            loss_each = F.cross_entropy(logits, labels_batch, weight=class_weights, reduction='none')
            loss = (loss_each * row_weights).mean()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        total_loss += float(loss.detach())
        if step % 500 == 0:
            print(f'epoch {epoch+1}/{EPOCHS}, step {step}/{len(train_loader)}, loss={total_loss/step:.4f}')
    print(f'epoch {epoch+1}/{EPOCHS} complete; mean loss={total_loss/len(train_loader):.4f}')

In [ ]:
# Predict all test passages and write the required Kaggle code-competition file.
test_ds = TextDataset(test_texts)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2,
                         pin_memory=torch.cuda.is_available(), collate_fn=collator)
model.eval()
pred_ids = []
with torch.no_grad():
    for batch in test_loader:
        inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = model(**inputs, return_dict=False)[0]
        pred_ids.extend(logits.argmax(dim=1).cpu().numpy().tolist())

submission = pd.DataFrame({'id': test_df['id'], 'model': le.inverse_transform(np.asarray(pred_ids))})
if submission['id'].duplicated().any() or submission.isna().any().any():
    raise ValueError('Submission has duplicate IDs or missing values')
if set(submission['model']) - set(valid_models):
    raise ValueError('Submission includes an invalid label')
submission.to_parquet('/kaggle/working/submission.parquet', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.parquet, rows:', len(submission))